In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
from statsmodels.stats.sandwich_covariance import cov_hac

def stargazer_HC(models, type_out='text', type_HC='HC1', omit_stats=None, out=None, keep=None):
    """
    Generate regression tables with robust standard errors.

    Args:
    models (list): List of statsmodels regression results.
    type_out (str): Output type ("text", "latex", "html").
    type_HC (str): Type of robust covariance matrix estimator. Options include "HC0", "HC1", "HC2", "HC3".
    omit_stats (list): Statistics to omit from the table.
    out (str): File path to save the output, None to display.
    keep (list): Parameters to keep in the output.
    
    Returns:
    None
    """
    if not isinstance(models, list):
        models = [models]

    # Adjust standard errors
    for model in models:
        robust_cov = model.get_robustcov_results(cov_type=type_HC)
        model.cov_params_default = robust_cov.cov_params()

    stargazer = Stargazer(models)

    if omit_stats:
        stargazer.omit_statistics(omit_stats)
    if keep:
        stargazer.show_only(keep)
    
    stargazer.add_custom_notes(["Robust standard errors in parentheses"])
    
    if type_out == 'html':
        output = stargazer.render_html()
    elif type_out == 'latex':
        output = stargazer.render_latex()
    else:
        output = stargazer.render_text()

    if out:
        with open(out, 'w') as file:
            file.write(output)
    else:
        print(output)

# Example usage
if __name__ == '__main__':
    # Load data and fit models
    data = sm.datasets.get_rdataset('mtcars').data
    model1 = smf.ols('mpg ~ hp', data=data).fit()
    model2 = smf.ols('mpg ~ hp + wt', data=data).fit()

    # Create a list of models
    models = [model1, model2]

    # Call function
    stargazer_HC(models, type_out='text', type_HC='HC1', omit_stats=['LL', 'AIC', 'BIC'])
